### Supplement 2: Structured outputs (`2-structured.py`)

Get the model's answer as data in a shape you define, instead of free text.

**The key point:** You describe the shape as a Pydantic class. `parse()` sends it to OpenAI as a JSON Schema, the model is forced to write JSON that matches it, and the SDK turns that JSON back into an instance of your class at `message.parsed`.

```
class CalendarEvent(BaseModel)       written by you
   │  parse(..., response_format=CalendarEvent)
   │  the SDK turns the class into a JSON Schema and sends it
   ▼
the model writes JSON that matches the schema
   │  the SDK turns that JSON into a CalendarEvent
   ▼
message.content    the JSON text                (str)
message.parsed     the same data as an object   (CalendarEvent)
```

#### 1. Setup
Same as Supplement 1, plus `BaseModel` from Pydantic. `OpenAI(api_key=os.getenv("OPENAI_API_KEY"))` is the long form of Supplement 1's `OpenAI()`: Both read the key from `os.environ`, so they build the same client.

In [1]:
import os

from openai import OpenAI
from pydantic import BaseModel
from dotenv import load_dotenv

import json  # added for the prints below

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

#### 2. The shape: `CalendarEvent`
A Pydantic class: each line declares a field and its type. The model never sees this Python code. The SDK calls the class's `model_json_schema()` (a Pydantic method, printed below) to get a JSON Schema, tightens it for strict mode, and sends that. Here, tightening only adds `"additionalProperties": false`, which forbids extra keys. `list[str]` becomes `"type": "array"` with string `items`.

- **Who uses the schema:** OpenAI's server, not your code. It turns the schema into rules for which tokens the model may write (constrained decoding), so the model can't write a wrong key, a wrong type or a missing field.
- **`Field(description=...)` reaches the model:** Its text goes into the schema next to the field, so it works as an instruction for that field. `CalendarEvent` has none; `WeatherResponse` in Supplement 3 does.
- **Strict mode wants every field required:** For a value that may be missing, declare it as `str | None`, so the model can write `null` for it.

In [2]:
class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

In [3]:
print(json.dumps(CalendarEvent.model_json_schema(), indent=2))

{
  "properties": {
    "name": {
      "title": "Name",
      "type": "string"
    },
    "date": {
      "title": "Date",
      "type": "string"
    },
    "participants": {
      "items": {
        "type": "string"
      },
      "title": "Participants",
      "type": "array"
    }
  },
  "required": [
    "name",
    "date",
    "participants"
  ],
  "title": "CalendarEvent",
  "type": "object"
}


#### 3. The API call: `parse()`
`parse()` is `create()` plus two jobs the SDK does for you: it puts the schema for `response_format=CalendarEvent` into the request, and it turns the model's JSON reply back into a `CalendarEvent`. The type below prints as `ParsedChatCompletion[TypeVar]`; that label is cosmetic, and step 4 shows the parsed value really is a `CalendarEvent`.

- **Without `parse()`:** With `create()` you'd write the schema dict yourself, get the JSON back as text in `content`, and build the object yourself: `CalendarEvent(**json.loads(content))`.
- **JSON mode is the weaker option:** `response_format={"type": "json_object"}` only promises valid JSON, not your fields. Strict mode with a schema, which `parse()` sets up for you, promises the shape.

In [4]:
completion = client.chat.completions.parse(
    model="gpt-5-nano",
    messages=[
        {"role": "system", "content": "Extract the event information."},
        {
            "role": "user",
            "content": "Alice and Bob are going to a science fair on Friday.",
        },
    ],
    response_format=CalendarEvent,
)

In [5]:
print(type(completion))
# warnings=False hides a harmless Pydantic warning about the `parsed` field; it is still printed.
print(completion.model_dump_json(indent=2, warnings=False))

<class 'openai.types.chat.parsed_chat_completion.ParsedChatCompletion[TypeVar]'>
{
  "id": "chatcmpl-EUYdtnCze0W1l9hLyTCAWQMJNA3j6",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "{\"name\":\"Science Fair\",\"date\":\"Friday\",\"participants\":[\"Alice\",\"Bob\"]}",
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": null,
        "parsed": {
          "name": "Science Fair",
          "date": "Friday",
          "participants": [
            "Alice",
            "Bob"
          ]
        }
      }
    }
  ],
  "created": 1790951713,
  "model": "gpt-5-nano-2025-08-07",
  "object": "chat.completion",
  "metadata": null,
  "moderation": null,
  "service_tier": "default",
  "system_fingerprint": null,
  "usage": {
    "completion_tokens": 542,
    "prompt_tokens": 90,
    "total_tokens": 632,
    "com

#### 4. `content` vs. `parsed`
The message holds the same answer twice:
- `content` is the JSON **text** the model wrote, a `str`.
- `parsed` is that text turned into a `CalendarEvent` **object** by the SDK.

If the model refuses, `parsed` is `None` and `refusal` holds its explanation. So check `message.refusal` before you use `parsed`.

In [6]:
message = completion.choices[0].message

print("content:", message.content)
print(type(message.content))
print()
print("parsed:", message.parsed)
print(type(message.parsed))
print()
print("refusal:", message.refusal)

content: {"name":"Science Fair","date":"Friday","participants":["Alice","Bob"]}
<class 'str'>

parsed: name='Science Fair' date='Friday' participants=['Alice', 'Bob']
<class '__main__.CalendarEvent'>

refusal: None


#### 5. Using the fields
`event` is an ordinary Pydantic object: reach each field with a dot. These are the last four lines of `2-structured.py`, unchanged.

In [7]:
event = completion.choices[0].message.parsed
print(event.name)
print(event.date)
print(event.participants)

Science Fair
Friday
['Alice', 'Bob']


#### In short: Which way the data moves
These names blur together because they all involve JSON, but each one moves data in one direction. A schema is the mould and a dump is the casting: `model_json_schema()` works on the class, before any event exists, while `model_dump()` needs a real `event`.

<pre style="font-size: 0.85em; line-height: 1.35; white-space: pre; overflow-x: auto;">
The shape, sent to the model by parse():
  CalendarEvent.model_json_schema()   class      → schema (a dict)

Your data, for your code, a file or another service:
  event.model_dump()                  object     → dict
  event.model_dump_json()             object     → JSON text

Plain Python, no Pydantic:
  json.loads(s)                       JSON text  → dict
  json.dumps(d)                       dict       → JSON text

What the model sent back:
  message.content                     the JSON text it wrote
  message.parsed                      the same data as a CalendarEvent
</pre>

`model_dump()` works on the whole `completion` too, because the SDK's response classes are Pydantic models as well. That's the `model_dump_json()` in step 3.